In [24]:
from Crypto.Util.number import *
from sage.all import *
flag = b'nex{we_can_learn_a_lot_from_my_old_friend_crc}'

def old_friend(data, poly = 0x883d9fe55bba9af41f27bd6e0b0d8f8f):
    t = (1 << 128) - 1
    for b in data:
        t ^= b
        for _ in range(8):
            t = (t >> 1) ^ (poly & -(t & 1))
    return t ^ ((1 << 128) - 1)

def equivalent_affine_crc(crc = old_friend,crc_bits = 128, target_bytes = 46):
    zero_crc = crc(target_bytes*b"\x00")
    target_bits = 8 * target_bytes
    v2n = lambda v: int(''.join(map(str, v)), 2)
    n2v = lambda n: vector(GF(2), bin(n)[2:].zfill(crc_bits))
    # n2v_t = lambda n: vector(GF(2), bin(n)[2:].zfill(target_bits))
    Affine_Matrix = []
    for i in range(target_bits):
        v = vector(GF(2), (j == i for j in range(target_bits)))
        value = crc(long_to_bytes(v2n(v),target_bytes)) ^ zero_crc
        Affine_Matrix.append(n2v(value))
    # crc affine function: crc_128(x) = M*x+ C
    return matrix(GF(2),Affine_Matrix).transpose(), n2v(zero_crc)

def crc_reverse(crc_value):
    M , C = equivalent_affine_crc()
    v2n = lambda v: int(''.join(map(str, v)), 2)
    n2v = lambda n: vector(GF(2), bin(n)[2:].zfill(48)) #自行填充
    res = M.solve_right(n2v(crc_value)+C)
    return long_to_bytes(v2n(res),16)

crc_reverse(old_friend(flag))

TypeError: unsupported operand parent(s) for +: 'Vector space of dimension 125 over Finite Field of size 2' and 'Vector space of dimension 128 over Finite Field of size 2'

In [23]:
len(flag)

46

In [17]:
from Crypto.Util.number import *
getPrime(128*3)

23845339094541595645440262477424666641664696215014601332072563940008409489310889136227706813980900002216491535583329

In [89]:
def f(x):
    x ^= ((x >> 11) & 0xb451411bb451411b)
    x ^= ((x << 13) & 0x451411cc451411cc)
    x ^= ((x >> 17) & 0xaa191981aa191981)
    x ^= ((x >> 19) & 0xb1919810b1919810)
    x ^= ((x << 23) & 0x451411cd451411cd)
    x ^= ((x >> 29) & 0xb451411ab451411a)
    x ^= ((x << 31) & 0x451411cb451411cb)
    x ^= ((x << 37) & 0xaa19198caa19198c)
    x ^= ((x >> 41) & 0xb191981db191981d)
    x ^= ((x << 43) & 0x451411ce451411ce)

    x ^= ((x >> 10) & 0xb451411bb451411b)
    x ^= ((x << 12) & 0x451411cc451411cc)
    x ^= ((x >> 14) & 0xaa191981aa191981)
    x ^= ((x >> 16) & 0xb1919810b1919810)
    x ^= ((x << 20) & 0x451411cd451411cd)
    x ^= ((x >> 24) & 0xb451411ab451411a)
    x ^= ((x << 26) & 0x451411cb451411cb)
    x ^= ((x << 28) & 0xaa19198caa19198c)
    x ^= ((x >> 30) & 0xb191981db191981d)
    x ^= ((x << 32) & 0x451411ce451411ce)

    return x 

def calc(n):
    x = 0x1122334455667788
    for _ in range(n):
        x = f(x)
    return x

T = []
for i in range(128):
    x = 1<<(127-i)
    x = f(x)
    T.append(Integer(x).digits(2,padto = 128)[::-1])
T = matrix(GF(2),T)
x = 0x1122334455667788
b = matrix(ZZ,[ int(i) for i in bin(x)[2:].rjust(128,'0')])
c = int(''.join([str(int(i)) for i in (b*T**100000)[0]]),2)
print('nex{%s}' % hex(c)[2:])

nex{783fe3911d36ef09}


In [86]:
calc(100000)

8664894420484026121

6135716919169722009

In [5]:
maze = [0x0, 0x1, 0x0, 0x0, 0x0, 0x1, 0x0, 0x1, 0x0, 0x0, 0x0, 0x1, 0x0, 0x1, 0x0, 0x1, 0x0, 0x1, 0x0, 0x1, 0x0, 0x0, 0x0, 0x1, 0x0, 0x0, 0x0, 0x0, 0x0, 0x1, 0x1, 0x1, 0x1, 0x1, 0x1, 0x1, 0x0, 0x1, 0x0, 0x1, 0x0, 0x0, 0x0, 0x0, 0x0, 0x0, 0x0, 0x1, 0x0, 0x0, 0x0, 0x1, 0x1, 0x1, 0x1, 0x1, 0x1, 0x1, 0x1, 0x0, 0x0, 0x0, 0x0, 0x0, 0x0, 0x1, 0x0, 0x0, 0x0, 0x0, 0x1, 0x1, 0x0, 0x1, 0x0, 0x1, 0x1, 0x1, 0x1, 0x1, 0x0, 0x0, 0x0, 0x1, 0x0, 0x0, 0x0, 0x0, 0x0, 0x0, 0x1, 0x1, 0x0, 0x1, 0x0, 0x1, 0x1, 0x1, 0x1, 0x0, 0x0, 0x0, 0x0, 0x0]

from collections import deque

def bfs(maze, start, end):
    directions = [(1, 0, 'D'), (0, 1, 'S'), (0, -1, 'W'), (-1, 0, 'A')]  # 右 下 上 左
    visited = set()
    queue = deque([start])
    path = {start: []}  # 记录每个点到达的路径

    while queue:
        x, y = queue.popleft()

        if (x, y) == end:
            return path[end]  # 返回到达终点的路径

        for dx, dy, direction in directions:
            nx, ny = x + dx, y + dy

            if 0 <= nx <= 9 and 0 <= ny <= 9  and (nx, ny) not in visited:
                t = maze[10*ny + nx]
                if t == 0:
                    visited.add((nx, ny))
                    queue.append((nx, ny))
                    path[(nx, ny)] = path[(x, y)] + [direction]

    return None  # 如果没有找到路径，则返回 None

# 设置起点和终点
start = (0, 0)
end = (9, 9)

# 运行 BFS
path = bfs(maze, start, end)
if path is not None:
    print("Path:", ''.join(path))
else:
    print("No path found.")

path = ''.join(path)
print(path)

Path: SSDDWWDDSSDDSSAAAAAASSDDDDSSDDDDDS
SSDDWWDDSSDDSSAAAAAASSDDDDSSDDDDDS


In [10]:
from base64 import *
b64decode(b"tsgmqb9votEssHopXnkLn1wst1gLobYBn1oCxcMtrc5rpD==")

b'\xb6\xc8&\xa9\xbfo\xa2\xd1,\xb0z)^y\x0b\x9f\\,\xb7X\x0b\xa1\xb6\x01\x9fZ\x02\xc5\xc3-\xad\xcek\xa4'

In [15]:
1048629/3

349543.0

In [17]:
from gmpy2 import isqrt
isqrt(349543)

mpz(591)

'0x24f'